<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# FABnet IPv6 Network: Automatic Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** Creates a two-node experiment spanning two FABRIC sites and connects them using the **FABnet IPv6** service with automatic IP address and route configuration. FABnet is FABRIC's private Layer 3 network that interconnects all sites over high-performance links.

This is the **auto** approach -- you tell FABlib to configure IP addresses automatically at slice submission time, but you still create the networks and interfaces yourself.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create an IPv6 Layer 3 network (`type='IPv6'`) connecting nodes on different FABRIC sites
2. Use **auto mode** (`iface.set_mode('auto')`) to have FABlib configure IPv6 addresses during the post-boot phase
3. Add inter-site routes so nodes on different FABnet subnets can reach each other
4. Verify IPv6 connectivity between sites using `ping`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with creating basic slices -- see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb)

**Tip -- Three configuration approaches:** FABRIC offers three ways to set up FABnet IPv6:

| Approach | Notebook | You create networks? | You assign IPs? |
|----------|----------|---------------------|-----------------|
| **Full Auto** | [full_auto](create_l3network_fabnet_ipv6_full_auto.ipynb) | No (`add_fabnet()`) | No |
| **Auto** (this notebook) | You are here | Yes | No (auto mode) |
| **Manual** | [manual](create_l3network_fabnet_ipv6_manual.ipynb) | Yes | Yes |

</div>

## Background: FABnet IPv6 with Auto Mode

FABRIC provides a pair of Layer 3 IP networking services across every site: **FABnetv4** (IPv4) and **FABnetv6** (IPv6). These act as a private internet connecting experiments across the testbed.

With **auto mode**, you create the networks and attach interfaces yourself, but FABlib automatically assigns IPv6 addresses and configures the interfaces during the post-boot phase. You still need to add routes manually at slice definition time using `node.add_route()`.


Each site gets its own subnet. FABlib assigns addresses from the subnet and provides a gateway. The `add_route()` call tells each node how to reach the *other* site's subnet via the local gateway.

**NIC component model options:**

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected via FABNetv6.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import Python IP address utilities and FABlib
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select two **different** random sites so our nodes are placed on separate FABRIC locations, requiring traffic to traverse the backbone. The `get_random_sites()` method guarantees the returned sites are distinct.

In [ ]:
# Name for this experiment slice
slice_name = 'MySlice'

# Pick two distinct random FABRIC sites
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node names
node1_name = 'Node1'
node2_name = 'Node2'

# Each site needs its own FABnet IPv6 network
network1_name = 'net1'
network2_name = 'net2'

## Step 3: Build and Submit the Slice

The key steps in auto mode are:

1. **Create the networks** with `add_l3network(type='IPv6')` -- one per site
2. **Add NICs** to each node and get their interfaces
3. **Set interfaces to auto mode** with `iface.set_mode('auto')` -- FABlib will assign IPv6 addresses during post-boot
4. **Add routes** so each node knows how to reach the other site's subnet via the local gateway
5. **Submit** the slice

<div class="fab-danger">

**Important:** All interfaces attached to a single `l3network` must be on the **same** FABRIC site. That is why we create one network per site and add routes between them.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Create one FABnet IPv6 network per site ---
net1 = slice.add_l3network(name=network1_name, type='IPv6')
net2 = slice.add_l3network(name=network2_name, type='IPv6')

# --- Node1 (site1) ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a basic NIC and get its single interface
iface1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Enable auto mode: FABlib assigns an IPv6 address during post-boot
iface1.set_mode('auto')
# Attach this interface to the site1 network
net1.add_interface(iface1)
# Add a route so Node1 can reach net2's subnet via net1's gateway
node1.add_route(subnet=net2.get_subnet(), next_hop=net1.get_gateway())

# --- Node2 (site2) ---
node2 = slice.add_node(name=node2_name, site=site2)
# Add a basic NIC and get its single interface
iface2 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Enable auto mode for this interface as well
iface2.set_mode('auto')
# Attach this interface to the site2 network
net2.add_interface(iface2)
# Add a route so Node2 can reach net1's subnet via net2's gateway
node2.add_route(subnet=net1.get_subnet(), next_hop=net2.get_gateway())

# Submit the slice -- blocks until provisioning is complete (~2-5 min)
slice.submit();

<div class="fab-success">

**What just happened?** FABRIC provisioned two VMs on different sites, created FABnet IPv6 networks at each site, assigned IPv6 addresses automatically to each interface, and configured the routes you specified. The slice is ready to use.

</div>

---

## Step 4: Run the Experiment

With auto mode, the slice is ready for experimentation as soon as it becomes active. We will verify connectivity by pinging Node2 from Node1 across the FABRIC backbone.

<div class="fab-warning">

**Tip:** Auto mode works well when saving slices to a file and re-instantiating them later. Configuration tasks are stored in the saved slice, simplifying re-deployment.

</div>

In [ ]:
# Retrieve the slice (useful if returning to this notebook later)
slice = fablib.get_slice(slice_name)

# Get node objects
node1 = slice.get_node(name=node1_name)
node2 = slice.get_node(name=node2_name)

# Get Node2's IPv6 address from its interface on net2
node2_addr = node2.get_interface(network_name=network2_name).get_ip_addr()

# Ping Node2 from Node1 across the FABRIC backbone
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Slice stuck in `Configuring` | A site may be busy or temporarily unavailable | Try different sites by re-running the `get_random_sites()` cell |
| `ping` fails between nodes | Routes not configured correctly | Verify that `add_route()` was called with the correct subnet and gateway |
| `ping` shows `Network unreachable` | Interface not configured | Check that `set_mode('auto')` was called before `submit()` |
| No IPv6 address on interface | Auto configuration failed during post-boot | SSH into the node and run `ip -6 addr show` to debug; try resubmitting |
| `PDP Authorization check failed` | Project permissions issue | Contact your project lead or FABRIC support |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.get_random_sites(count)` | Select distinct random sites | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `slice.add_l3network(name, type)` | Add a Layer 3 network to the slice | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `node.add_component(model, name)` | Add a NIC or other component to a node | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `component.get_interfaces()` | Get the list of interfaces on a component | [get_interfaces](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.get_interfaces) |
| `iface.set_mode('auto')` | Enable automatic IP configuration | [set_mode](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_mode) |
| `network.add_interface(iface)` | Attach an interface to a network | [add_interface](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.add_interface) |
| `node.add_route(subnet, next_hop)` | Add a static route on the node | [add_route](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_route) |
| `network.get_subnet()` | Get the subnet assigned to the network | [get_subnet](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_subnet) |
| `network.get_gateway()` | Get the gateway IP for the network | [get_gateway](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_gateway) |
| `iface.get_ip_addr()` | Get the IP address assigned to the interface | [get_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_ip_addr) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FABnet IPv6 Full Auto** | [full_auto](create_l3network_fabnet_ipv6_full_auto.ipynb) | Simplest approach -- `add_fabnet()` handles everything |
| **FABnet IPv6 Manual** | [manual](create_l3network_fabnet_ipv6_manual.ipynb) | Full control over IP address assignment |
| **FABnet IPv4 Auto** | [ipv4_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Same pattern but with IPv4 addressing |
| **FABnet IPv6 Ext** | [ipv6_ext](../create_l3network_fabnet_ipv6ext_manual/create_l3network_fabnet_ipv6ext_manual.ipynb) | IPv6 with external (public) connectivity |
| **Sub Interfaces** | [sub_interfaces](../sub_interfaces/sub_interfaces.ipynb) | Multiple virtual interfaces on a single NIC |